# 第10章: 事前学習済み言語モデル（GPT型）

GPT型（Transformerのデコーダ型）の事前学習済みモデルを用いて、言語生成・評判分析・ファインチューニング・強化学習に取り組む。

## 90. 次単語予測

“The movie was full of” に続くトークン（トークン**列**ではなく一つのトークン）として適切なもの上位10個と、その確率（尤度）を求める。あわせて、プロンプトがどのようなトークン列に変換されたかを確認する。

### セットアップ
macOS + miniforge 環境では OpenMP の重複初期化エラー（OMP #15）が出るため、`KMP_DUPLICATE_LIB_OK` を有効にしておく。

In [8]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"        # macOS の OpenMP 重複初期化を回避
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "gpt2"
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
model.eval()
print("loaded:", MODEL_NAME)

device: mps


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 11444.42it/s]


loaded: gpt2


### プロンプトのトークン化を確認
GPT-2 は byte-level BPE を使う。単語先頭の空白は `Ġ`（表示上は `Ġ`）としてトークンに含まれる点に注意する。

In [9]:
prompt = "The movie was full of"
enc = tokenizer(prompt, return_tensors="pt")
input_ids = enc["input_ids"][0]

print("prompt      :", repr(prompt))
print("input_ids   :", input_ids.tolist())
print("tokens (BPE):", tokenizer.convert_ids_to_tokens(input_ids.tolist()))
print("decoded each:", [tokenizer.decode([i]) for i in input_ids.tolist()])

prompt      : 'The movie was full of'
input_ids   : [464, 3807, 373, 1336, 286]
tokens (BPE): ['The', 'Ġmovie', 'Ġwas', 'Ġfull', 'Ġof']
decoded each: ['The', ' movie', ' was', ' full', ' of']


### 次の1トークンの予測（上位10件と確率）
最後の位置のロジットに softmax を適用し、次トークンの確率分布を得て上位10件を取り出す。

In [10]:
with torch.no_grad():
    logits = model(**enc.to(device)).logits      # shape: (1, seq_len, vocab_size)

next_token_logits = logits[0, -1]                # 最後のトークン位置 = 次トークンの予測
probs = torch.softmax(next_token_logits, dim=-1) # 尤度（確率）に変換

topk = torch.topk(probs, 10)

print(f"次トークンの上位10候補: {prompt!r} __")
print("-" * 40)
for rank, (p, idx) in enumerate(zip(topk.values.tolist(), topk.indices.tolist()), 1):
    token = tokenizer.decode([idx])
    print(f"{rank:2d}. {token!r:15s} (id={idx:6d})  p={p:.4f}")

次トークンの上位10候補: 'The movie was full of' __
----------------------------------------
 1. ' jokes'        (id= 14532)  p=0.0219
 2. ' great'        (id=  1049)  p=0.0186
 3. ' laughs'       (id= 22051)  p=0.0115
 4. ' bad'          (id=  2089)  p=0.0109
 5. ' surprises'    (id= 24072)  p=0.0107
 6. ' references'   (id= 10288)  p=0.0105
 7. ' fun'          (id=  1257)  p=0.0100
 8. ' humor'        (id= 14733)  p=0.0074
 9. ' "'            (id=   366)  p=0.0074
10. ' the'          (id=   262)  p=0.0067


## 91. 続きのテキストの予測

“The movie was full of” に続くテキストを、**デコーディング方法**（greedy / beam / サンプリング）と**温度パラメータ temperature** を変えながら複数生成し、出力の変化を観察する。

※ 90 で読み込んだ `tokenizer` / `model` をそのまま用いる。

In [11]:
model.config.pad_token_id = tokenizer.eos_token_id
prompt = "The movie was full of"
enc = tokenizer(prompt, return_tensors="pt").to(device)

def generate(seed=0, **kwargs):
    torch.manual_seed(seed)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=30,
                             pad_token_id=tokenizer.eos_token_id, **kwargs)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print("[greedy]            ", generate(do_sample=False))
print()
print("[beam search (5)]   ", generate(num_beams=5, do_sample=False))
print()
for t in (0.7, 1.0, 1.5):
    print(f"[sampling  T={t}]    ", generate(do_sample=True, temperature=t, top_k=0))
    print()
print("[top-k = 50]        ", generate(do_sample=True, top_k=50))
print()
print("[top-p = 0.9]       ", generate(do_sample=True, top_p=0.9, top_k=0))

[greedy]             The movie was full of jokes and jokes about how the movie was a joke. It was a joke about how the movie was a joke. It was a joke about how the

[beam search (5)]    The movie was full of jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and jokes and

[sampling  T=0.7]     The movie was full of mementos of the beloved characters, but there are several scenes that took place in the same city, a scene that was specifically mentioned in the script

[sampling  T=1.0]     The movie was full of chaos and was just too good to get any one of its protagonists alive. Below are 16 lesser-known characters the best of the rest of the cast

[sampling  T=1.5]     The movie was full of chaos but was just too stayed-at-your-facts bullshit musing desperate Americans hated whore stuff water whilst the best Americans be Westminster chaerbates

[top-k = 50]         The movie was full of some question

## 92. 予測されたテキストの確率を計算

続きのテキストを生成し、**生成された各トークンの尤度（確率）** を表示する。出力が長いと読みにくいので `max_new_tokens` で短く打ち切る。

In [12]:
torch.manual_seed(0)
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=15,
                         do_sample=True, top_p=0.9, top_k=0,
                         pad_token_id=tokenizer.eos_token_id,
                         return_dict_in_generate=True, output_scores=True)

prompt_len = enc["input_ids"].shape[1]
gen_ids = out.sequences[0, prompt_len:]

print("生成:", tokenizer.decode(gen_ids, skip_special_tokens=True))
print("-" * 40)
logp_sum = 0.0
for score, tid in zip(out.scores, gen_ids):
    probs = torch.softmax(score[0], dim=-1)   # そのステップの確率分布
    p = probs[tid].item()                     # 実際に選ばれたトークンの尤度
    logp_sum += torch.log(probs[tid]).item()
    print(f"  {tokenizer.decode([tid])!r:15s} p={p:.4f}")
print("-" * 40)
print(f"系列全体の対数尤度 sum(log p) = {logp_sum:.3f}")

生成:  chaos and was just too good to get any one of its protagonists alive.
----------------------------------------
  ' chaos'        p=0.0004
  ' and'          p=0.3235
  ' was'          p=0.0072
  ' just'         p=0.0071
  ' too'          p=0.0365
  ' good'         p=0.0695
  ' to'           p=0.7036
  ' get'          p=0.0068
  ' any'          p=0.0669
  ' one'          p=0.0125
  ' of'           p=0.3923
  ' its'          p=0.1108
  ' protagonists' p=0.0030
  ' alive'        p=0.0059
  '.'             p=0.7467
----------------------------------------
系列全体の対数尤度 sum(log p) = -51.516


## 93. パープレキシティ

パープレキシティ $\mathrm{PPL} = \exp\!\big(-\frac{1}{N}\sum_i \log p(w_i \mid w_{<i})\big)$ は、モデルが文をどれだけ「驚かず」予測できるかの指標（小さいほど自然）。`labels=input_ids` を渡すと平均交差エントロピー損失が得られ、その `exp` が PPL。

文法的に正しい2文と、故意に主語・動詞の一致を崩した2文を比較する。

In [13]:
import math

def perplexity(text):
    ids = tokenizer(text, return_tensors="pt").to(device)["input_ids"]
    with torch.no_grad():
        loss = model(ids, labels=ids).loss   # 平均交差エントロピー（自然対数）
    return math.exp(loss.item())

sentences = [
    "The movie was full of surprises",   # 正
    "The movies were full of surprises", # 正
    "The movie were full of surprises",  # 誤（were）
    "The movies was full of surprises",  # 誤（was）
]
for s in sentences:
    print(f"  PPL = {perplexity(s):7.2f}  | {s}")

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


  PPL =   99.36  | The movie was full of surprises
  PPL =  126.48  | The movies were full of surprises
  PPL =  278.88  | The movie were full of surprises
  PPL =  274.66  | The movies was full of surprises


## 94. チャットテンプレート

90〜93 で使った素の `gpt2` は指示追従用にチューニングされておらず、チャットテンプレートも持たない。ここからは **instruct（chat）モデル** に切り替える。本ノートでは軽量でゲート無しの **`Qwen/Qwen2.5-0.5B-Instruct`** を用いる（16GB・MPSで 94〜99 を通せる大きさ）。

“What do you call a sweet eaten after dinner?” に応答させるため、`tokenizer.apply_chat_template` で**言語モデルに与えるプロンプト**を作り、その中身を確認する。

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM

CHAT_MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
chat_tok = AutoTokenizer.from_pretrained(CHAT_MODEL)
chat_model = AutoModelForCausalLM.from_pretrained(
    CHAT_MODEL, torch_dtype=torch.float32).to(device).eval()
print("loaded:", CHAT_MODEL)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 452.42it/s]


loaded: Qwen/Qwen2.5-0.5B-Instruct


In [15]:
messages = [
    {"role": "user", "content": "What do you call a sweet eaten after dinner?"},
]

# tokenize=False で「モデルに与えられる文字列」を確認する
prompt_str = chat_tok.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True)
print("=== chat template を適用したプロンプト ===")
print(prompt_str)

=== chat template を適用したプロンプト ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What do you call a sweet eaten after dinner?<|im_end|>
<|im_start|>assistant



`<|im_start|>role ... <|im_end|>` という特殊トークンで system / user / assistant の各発話が区切られる（Qwen が採用する ChatML 形式）。末尾に `add_generation_prompt=True` で `<|im_start|>assistant\n` が付き、モデルはここから**アシスタントの発話を生成**する。

In [16]:
def chat_generate(messages, max_new_tokens=120):
    inputs = chat_tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(device)
    n_prompt = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = chat_model.generate(**inputs, max_new_tokens=max_new_tokens,
                                  do_sample=False,
                                  pad_token_id=chat_tok.eos_token_id)
    return chat_tok.decode(out[0, n_prompt:], skip_special_tokens=True)

response = chat_generate(messages)
print("=== 応答 ===")
print(response)

=== 応答 ===
The sweet eaten after dinner is called "souple." It's a type of sweet dessert that is often served as an accompaniment to meals and can be enjoyed at any time of the day or night. Souple typically consists of a sweet filling wrapped in a soft, chewy dough or pastry shell, which is then baked until it becomes tender and golden brown. It's a popular treat in many cultures around the world, particularly in Asia, where it's known as "souple" or "souple cake."


## 95. マルチターンのチャット

94 の応答を会話履歴に**assistant発話として追加**し、続けて“Please give me the plural form of the word with its spelling in reverse order.” と質問する。このとき言語モデルに与えられるプロンプト（履歴すべてを含む）を確認する。

In [17]:
# 94 の応答を履歴に積み、さらにユーザ発話を追加
messages.append({"role": "assistant", "content": response})
messages.append({"role": "user",
                 "content": "Please give me the plural form of the word "
                            "with its spelling in reverse order."})

# モデルに与えられるプロンプト（マルチターン）を確認
prompt_str2 = chat_tok.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True)
print("=== マルチターンのプロンプト ===")
print(prompt_str2)

=== マルチターンのプロンプト ===
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What do you call a sweet eaten after dinner?<|im_end|>
<|im_start|>assistant
The sweet eaten after dinner is called "souple." It's a type of sweet dessert that is often served as an accompaniment to meals and can be enjoyed at any time of the day or night. Souple typically consists of a sweet filling wrapped in a soft, chewy dough or pastry shell, which is then baked until it becomes tender and golden brown. It's a popular treat in many cultures around the world, particularly in Asia, where it's known as "souple" or "souple cake."<|im_end|>
<|im_start|>user
Please give me the plural form of the word with its spelling in reverse order.<|im_end|>
<|im_start|>assistant



In [18]:
response2 = chat_generate(messages)
print("=== 追質問への応答 ===")
print(response2)

=== 追質問への応答 ===
Certainly! The plural form of the word "souple" would be "soups," which is spelled backwards as "soup."


## 96. プロンプトによる感情分析（SST-2 の正解率）

**ファインチューニングせず**、プロンプトだけで感情分析（ポジ/ネガ）を行う。SST-2 の開発データ（validation, 872件）に対し、各レビューをプロンプトに埋め込んでinstruct モデルに与え、正解率を測る。

ラベル: `0 = negative`, `1 = positive`。

In [19]:
from datasets import load_dataset

sst2 = load_dataset("stanfordnlp/sst2", split="validation")
print("SST-2 validation:", len(sst2), "件")
print("例:", sst2[0])   # {'idx':..., 'sentence':..., 'label': 1}

SST-2 validation: 872 件
例: {'idx': 0, 'sentence': "it 's a charming and often affecting journey . ", 'label': 1}


In [20]:
# ラベル語の先頭トークン ID（先頭スペース込み）
def first_token_id(word):
    return chat_tok(word, add_special_tokens=False)["input_ids"][0]

POS_ID = first_token_id(" positive")
NEG_ID = first_token_id(" negative")
print("positive ->", POS_ID, repr(chat_tok.decode([POS_ID])),
      "| negative ->", NEG_ID, repr(chat_tok.decode([NEG_ID])))

def predict_sentiment(text):
    messages = [{"role": "user", "content":
        "Is the sentiment of the following movie review positive or negative? "
        "Answer with a single word.\n"
        f"Review: {text.strip()}"}]
    inputs = chat_tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(device)
    with torch.no_grad():
        next_logits = chat_model(**inputs).logits[0, -1]
    # positive と negative のロジットを比較
    return 1 if next_logits[POS_ID] > next_logits[NEG_ID] else 0

# 動作確認
print(predict_sentiment(sst2[0]["sentence"]), "(正解", sst2[0]["label"], ")")

positive -> 6785 ' positive' | negative -> 8225 ' negative'
1 (正解 1 )


In [21]:
from tqdm.auto import tqdm

correct = 0
for ex in tqdm(sst2, desc="zero-shot SST-2"):
    if predict_sentiment(ex["sentence"]) == ex["label"]:
        correct += 1

accuracy = correct / len(sst2)
print(f"\nZero-shot 正解率 (SST-2 dev, {len(sst2)}件): {accuracy:.4f}  "
      f"({correct}/{len(sst2)})")

zero-shot SST-2: 100%|██████████| 872/872 [00:47<00:00, 18.29it/s]


Zero-shot 正解率 (SST-2 dev, 872件): 0.9163  (799/872)


## 97. 埋め込みに基づく感情分析

事前学習済みモデルでテキストを**ベクトルにエンコード**し、フィードフォワード層で極性ラベルを予測する。

- LM は**凍結**。最終隠れ層を attention マスク付き平均プーリング → 文ベクトル（896次元）
- その上に MLP 分類器を学習（train 3000件サブサンプル / dev 全872件で評価）

In [ ]:
# === 97 セットアップ===
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"      # macOS OpenMP 対策
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# 前のセクションのモデルが残っていれば解放（16GB クラッシュ対策）
for _n in ["model", "tokenizer", "chat_model", "chat_tok", "base",
           "ft_model", "dpo_base", "dpo_model", "enc_lm", "clf"]:
    globals().pop(_n, None)
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

device = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

enc_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
enc_lm = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32,
    output_hidden_states=True).to(device).eval()

train_ds = load_dataset("stanfordnlp/sst2", split="train").shuffle(seed=0).select(range(3000))
dev_ds   = load_dataset("stanfordnlp/sst2", split="validation")
print("device:", device, "| train:", len(train_ds), "| dev:", len(dev_ds))

[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 519.20it/s]


device: mps | train: 3000 | dev: 872


In [22]:
# テキストを文ベクトルにエンコード（数分かかる）
@torch.no_grad()
def encode_texts(texts, batch_size=32):
    """LM 最終隠れ層の masked mean pooling"""
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = [t.strip() for t in texts[i:i + batch_size]]
        enc = enc_tok(batch, return_tensors="pt", padding=True,
                      truncation=True, max_length=64).to(device)
        hidden = enc_lm(**enc).hidden_states[-1]           # (B, T, H)
        mask = enc["attention_mask"].unsqueeze(-1).float()
        vecs.append(((hidden * mask).sum(1) / mask.sum(1)).cpu())
    return torch.cat(vecs)

X_train = encode_texts(train_ds["sentence"]); y_train = torch.tensor(train_ds["label"])
X_dev   = encode_texts(dev_ds["sentence"]);   y_dev   = torch.tensor(dev_ds["label"])
print("X_train:", tuple(X_train.shape), "| X_dev:", tuple(X_dev.shape))

# ベクトル化が終わったら LM は不要 → 解放
del enc_lm
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

X_train: (3000, 896) | X_dev: (872, 896)


In [23]:
# フィードフォワード分類器の学習（CPU で十分軽い）
mu, sd = X_train.mean(0), X_train.std(0) + 1e-6      # 標準化
X_train_n, X_dev_n = (X_train - mu) / sd, (X_dev - mu) / sd

torch.manual_seed(0)
clf = nn.Sequential(
    nn.Linear(X_train.shape[1], 256), nn.ReLU(), nn.Dropout(0.2),
    nn.Linear(256, 2),
)
optimizer = torch.optim.AdamW(clf.parameters(), lr=1e-3, weight_decay=1e-4)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(30):
    clf.train()
    perm = torch.randperm(len(X_train_n))
    for i in range(0, len(X_train_n), 64):
        idx = perm[i:i + 64]
        optimizer.zero_grad()
        loss_fn(clf(X_train_n[idx]), y_train[idx]).backward()
        optimizer.step()
    if (epoch + 1) % 10 == 0:
        clf.eval()
        with torch.no_grad():
            acc = (clf(X_dev_n).argmax(1) == y_dev).float().mean().item()
        print(f"epoch {epoch+1:2d}: dev acc = {acc:.4f}")

clf.eval()
with torch.no_grad():
    dev_acc = (clf(X_dev_n).argmax(1) == y_dev).float().mean().item()
print(f"\n97 最終 dev 正解率: {dev_acc:.4f}")

epoch 10: dev acc = 0.8131
epoch 20: dev acc = 0.8039
epoch 30: dev acc = 0.8073

97 最終 dev 正解率: 0.8073


## 98. ファインチューニング

96 のプロンプトに対して**正解の感情ラベル（`positive`/`negative`）を応答として返す**ようモデルをファインチューニングする（SFT）。

- **LoRA**（peft）で少数パラメータのみ学習（16GB 向け・省メモリ）
- 損失は**応答トークンのみ**（プロンプト部分は `-100` でマスク）
- train 800件・1 epoch、評価は dev 全872件

In [ ]:
# === 98 セットアップ ===
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, random
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

# 前のセクションのモデル・ベクトルを解放
for _n in ["model", "tokenizer", "chat_model", "chat_tok", "enc_lm", "enc_tok",
           "clf", "dpo_base", "dpo_model",
           "X_train", "X_dev", "X_train_n", "X_dev_n"]:
    globals().pop(_n, None)
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

device = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

LABELS = {0: "negative", 1: "positive"}

def build_prompt(sentence):
    msgs = [{"role": "user", "content":
        "Is the sentiment of the following movie review positive or negative? "
        "Answer with a single word.\nReview: " + sentence.strip()}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

train_ds = load_dataset("stanfordnlp/sst2", split="train").shuffle(seed=0).select(range(800))
dev_ds   = load_dataset("stanfordnlp/sst2", split="validation")
print("device:", device, "| SFT train:", len(train_ds), "| dev:", len(dev_ds))

device: mps | SFT train: 800 | dev: 872


In [25]:
# 学習データ作成: プロンプト + 正解ラベル応答（損失は応答部分のみ）
def build_example(ex):
    prompt = build_prompt(ex["sentence"])
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    a_ids = tok(LABELS[ex["label"]] + tok.eos_token, add_special_tokens=False)["input_ids"]
    return p_ids + a_ids, [-100] * len(p_ids) + a_ids

sft_data = [build_example(ex) for ex in train_ds]
print("例) 応答部分:", tok.decode([t for t in sft_data[0][1] if t != -100]))

# ベースモデル + LoRA アダプタ
ft_model = get_peft_model(
    AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float32).to(device),
    LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05,
               target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))
ft_model.print_trainable_parameters()

例) 応答部分: negative<|im_end|>


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 753.37it/s]


trainable params: 540,672 || all params: 494,573,440 || trainable%: 0.1093


In [ ]:
# 学習（1 epoch・MPS で）
def collate(batch):
    maxlen = max(len(x[0]) for x in batch)
    pad = tok.pad_token_id
    ii = torch.tensor([x[0] + [pad]  * (maxlen - len(x[0])) for x in batch])
    ll = torch.tensor([x[1] + [-100] * (maxlen - len(x[1])) for x in batch])
    return ii.to(device), ll.to(device), (ii != pad).long().to(device)

optimizer = torch.optim.AdamW(ft_model.parameters(), lr=2e-4)
ft_model.train()
random.seed(0); random.shuffle(sft_data)
BS = 8
for i in range(0, len(sft_data), BS):
    input_ids, labels, attn = collate(sft_data[i:i + BS])
    optimizer.zero_grad()
    loss = ft_model(input_ids=input_ids, attention_mask=attn, labels=labels).loss
    loss.backward()
    optimizer.step()
    if (i // BS) % 20 == 0:
        print(f"step {i // BS:3d}: loss = {loss.item():.4f}")
print("training done.")

step   0: loss = 0.3588
step  20: loss = 0.0231
step  40: loss = 0.1279
step  60: loss = 0.0501
step  80: loss = 0.4098
training done.


In [ ]:
# 評価: positive / negative のロジット比較で dev 全件
ft_model.eval()
POS = tok("positive", add_special_tokens=False)["input_ids"][0]
NEG = tok("negative", add_special_tokens=False)["input_ids"][0]

@torch.no_grad()
def predict_ft(sentence):
    inputs = tok(build_prompt(sentence), return_tensors="pt").to(device)
    logits = ft_model(**inputs).logits[0, -1]
    return 1 if logits[POS] > logits[NEG] else 0

correct = sum(predict_ft(ex["sentence"]) == ex["label"]
              for ex in tqdm(dev_ds, desc="fine-tuned eval"))
print(f"\n98 ファインチューニング後 dev 正解率: "
      f"{correct / len(dev_ds):.4f} ({correct}/{len(dev_ds)})")

fine-tuned eval: 100%|██████████| 872/872 [01:05<00:00, 13.23it/s]


98 ファインチューニング後 dev 正解率: 0.9266 (808/872)


In [28]:
# 定性確認: 実際に応答を生成
sample = dev_ds[0]["sentence"]
inputs = tok(build_prompt(sample), return_tensors="pt").to(device)
with torch.no_grad():
    out = ft_model.generate(**inputs, max_new_tokens=3, do_sample=False,
                            pad_token_id=tok.eos_token_id)
print("Review:", sample.strip())
print("生成応答:", tok.decode(out[0, inputs["input_ids"].shape[1]:],
                          skip_special_tokens=True))
print("正解:", LABELS[dev_ds[0]["label"]])

Review: it 's a charming and often affecting journey .
生成応答: positive
正解: positive


## 99. 選好チューニング（DPO）

96 のプロンプトに対し、**正解ラベルの応答 = 望ましい (chosen)**、**誤ラベルの応答 = 望ましくない (rejected)** として選好チューニングを行う。アルゴリズムは **DPO**。


In [ ]:
# === 99 セットアップ ===
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from tqdm.auto import tqdm

# 前のセクションのモデルを解放
for _n in ["model", "tokenizer", "chat_model", "chat_tok", "enc_lm", "enc_tok",
           "clf", "ft_model", "base",
           "X_train", "X_dev", "X_train_n", "X_dev_n"]:
    globals().pop(_n, None)
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

device = "mps" if torch.backends.mps.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_NAME)

LABELS = {0: "negative", 1: "positive"}
POS = tok("positive", add_special_tokens=False)["input_ids"][0]
NEG = tok("negative", add_special_tokens=False)["input_ids"][0]

def build_prompt(sentence):
    msgs = [{"role": "user", "content":
        "Is the sentiment of the following movie review positive or negative? "
        "Answer with a single word.\nReview: " + sentence.strip()}]
    return tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)

train_ds = load_dataset("stanfordnlp/sst2", split="train").shuffle(seed=0).select(range(400))
dev_ds   = load_dataset("stanfordnlp/sst2", split="validation")
print("device:", device, "| 選好ペア:", len(train_ds), "| dev:", len(dev_ds))

device: mps | 選好ペア: 400 | dev: 872


In [30]:
# 選好ペアの作成
def preference_example(ex):
    prompt = build_prompt(ex["sentence"])
    chosen   = LABELS[ex["label"]]     + tok.eos_token   # 正解ラベル
    rejected = LABELS[1 - ex["label"]] + tok.eos_token   # 誤ラベル
    return prompt, chosen, rejected

pref_data = [preference_example(ex) for ex in train_ds]
print("例) chosen =", repr(pref_data[0][1]), "| rejected =", repr(pref_data[0][2]))

# ベース + LoRA（adapter on/off で方策/参照を切替）
dpo_model = get_peft_model(
    AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float32).to(device),
    LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0,
               target_modules=["q_proj", "v_proj"], task_type="CAUSAL_LM"))

def response_logprob(prompt, response):
    """response 部分のトークン対数尤度の和（現在の adapter 状態で計算）"""
    p_ids = tok(prompt,   add_special_tokens=False)["input_ids"]
    r_ids = tok(response, add_special_tokens=False)["input_ids"]
    ids = torch.tensor([p_ids + r_ids]).to(device)
    logprobs = F.log_softmax(dpo_model(ids).logits[0], dim=-1)
    total = 0.0
    for k, tid in enumerate(r_ids):
        total = total + logprobs[len(p_ids) + k - 1, tid]   # 直前位置が予測位置
    return total

例) chosen = 'negative<|im_end|>' | rejected = 'positive<|im_end|>'


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 710.89it/s]


In [31]:
# DPO 学習
BETA = 0.1
optimizer = torch.optim.AdamW(
    [p for p in dpo_model.parameters() if p.requires_grad], lr=1e-4)

dpo_model.train()
for step, (prompt, chosen, rejected) in enumerate(pref_data):
    # 方策（adapter 有効）
    pol_c = response_logprob(prompt, chosen)
    pol_r = response_logprob(prompt, rejected)
    # 参照（adapter 無効・勾配なし）
    with torch.no_grad(), dpo_model.disable_adapter():
        ref_c = response_logprob(prompt, chosen)
        ref_r = response_logprob(prompt, rejected)

    logits = BETA * ((pol_c - ref_c) - (pol_r - ref_r))
    loss = -F.logsigmoid(logits)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

    if step % 80 == 0:
        print(f"step {step:3d}: loss={loss.item():.4f}  "
              f"margin(chosen-rejected)={(pol_c - pol_r).item():.2f}")
print("DPO training done.")

step   0: loss=0.6931  margin(chosen-rejected)=5.03
step  80: loss=0.6732  margin(chosen-rejected)=1.68
step 160: loss=0.2182  margin(chosen-rejected)=19.12
step 240: loss=0.0677  margin(chosen-rejected)=32.33
step 320: loss=0.0800  margin(chosen-rejected)=28.70
DPO training done.


In [32]:
# 評価: dev 全件
dpo_model.eval()

@torch.no_grad()
def predict_dpo(sentence):
    inputs = tok(build_prompt(sentence), return_tensors="pt").to(device)
    logits = dpo_model(**inputs).logits[0, -1]
    return 1 if logits[POS] > logits[NEG] else 0

correct = sum(predict_dpo(ex["sentence"]) == ex["label"]
              for ex in tqdm(dev_ds, desc="DPO eval"))
print(f"\n99 DPO 後 dev 正解率: {correct / len(dev_ds):.4f} ({correct}/{len(dev_ds)})")

DPO eval: 100%|██████████| 872/872 [01:05<00:00, 13.22it/s]


99 DPO 後 dev 正解率: 0.9255 (807/872)
